# 手語翻譯 Transformer：Dataset → Encoder-Decoder → Forward Pass

這份 Notebook 將之前的 `dataset_loader.py`、`transformer_model.py` 和 `test_transformer.py` 合併成**一份可依序執行的 Jupyter Notebook**。

**本次目標**：讀取現有的 `train.json`、`val.json`、`test.json`、`vocab.json` 和各支影片的 `.npy`，建立小型 Transformer，並確認模型能成功輸出 `(Batch, 目標長度, Vocabulary Size)`。

> 目前是模型**前向傳播測試**，不是正式的 233 筆訓練流程。最後附上可選的單步反向傳播測試。

## 0. 執行前準備

建議把 Notebook 放在**專案根目錄**。如果從根目錄下的其他資料夾執行，程式也會嘗試向上尋找專案根目錄。

需要的套件：`numpy`、`torch`、`jupyter`（或 VS Code 的 Jupyter 擴充套件）。

預期檔案結構：

```text
專案根目錄/
├── rewrite_Phase3_train/
│   └── dataset/
│       ├── manifest.json
│       ├── train.json
│       ├── val.json
│       ├── test.json
│       └── vocab.json
└── yolo_pose_mediapie/
    └── hand_keypoints/
        ├── part_1_1.npy
        └── ...
```

**FPS 設定**：如果現有 `.npy` 是 30 FPS，就把 `FRAME_STEP` 設為 `2`，在讀取時降到約 15 FPS；如果 `.npy` 已是 15 FPS，設為 `1`。此方式預設原始資料採固定 30 FPS。

In [1]:
import json
import math
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

FRAME_STEP = 2       # 30 FPS → 約 15 FPS；若 NPY 已是 15 FPS，改成 1
BATCH_SIZE = 1
PAD_ID = 0

D_MODEL = 192
NHEAD = 4
ENCODER_LAYERS = 3
DECODER_LAYERS = 3
DIM_FF = 512
DROPOUT = 0.2
MAX_LEN = 4096

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / 'rewrite_Phase3_train' / 'dataset' / 'train.json').is_file()),
    Path.cwd()
)
DATASET_DIR = PROJECT_ROOT / 'rewrite_Phase3_train' / 'dataset'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('專案根目錄:', PROJECT_ROOT)
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

專案根目錄: c:\Users\User\Documents\Push-Github\sign-language
Device: cuda
GPU: NVIDIA GeForce RTX 5060 Ti


## 1. 讀取資料與建立 Dataset

每筆資料包含：

- `keypoints`：形狀 `(T, 128)`，每個 Frame 為左右手的 126 個座標加上 2 個存在狀態。
- `tokens`：形狀 `(L,)`，對應已建立 Vocabulary 的 `<bos>`、文字 Token、`<eos>`。

不同影片的 `T` 和 `L` 可以不一樣。

In [2]:
class SignLanguageDataset(Dataset):
    def __init__(self, json_path, project_root=PROJECT_ROOT, frame_step=FRAME_STEP):
        with open(json_path, 'r', encoding='utf-8') as f:
            self.rows = json.load(f)
        self.project_root = project_root
        self.frame_step = frame_step

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        item = self.rows[idx]
        npy_path = self.project_root / item['npy_path']
        keypoints = np.load(npy_path)[::self.frame_step].astype(np.float32, copy=True)
        tokens = np.asarray(item['token_ids'], dtype=np.int64)

        return torch.from_numpy(keypoints), torch.from_numpy(tokens)


train_dataset = SignLanguageDataset(DATASET_DIR / 'train.json')
val_dataset = SignLanguageDataset(DATASET_DIR / 'val.json')
test_dataset = SignLanguageDataset(DATASET_DIR / 'test.json')

with open(DATASET_DIR / 'vocab.json', 'r', encoding='utf-8') as f:
    vocab = json.load(f)

vocab_size = len(vocab)
print('Training:', len(train_dataset))
print('Validation:', len(val_dataset))
print('Test:', len(test_dataset))
print('Vocabulary Size:', vocab_size)

Training: 233
Validation: 29
Test: 29
Vocabulary Size: 1916


## 2. Dynamic Padding 與 Masks

`collate_fn` 將同一個 Batch 中的序列補到該 Batch 最長的長度：

- `src_key_padding_mask`：標記補齊的 Frame（`True` 表示忽略）。
- `tgt_key_padding_mask`：標記補齊的文字 Token。
- `causal_mask`：阻止 Decoder 在預測下一個 Token 時偷看後面的正確答案。

注意：Source Padding Mask **不是**每個 Frame 最後兩維的「左右手存在狀態」，兩者用途不同。

In [3]:
def collate_fn(batch):
    keypoints, tokens = zip(*batch)

    src_lengths = torch.tensor([len(kp) for kp in keypoints])

    src = pad_sequence(keypoints, batch_first=True, padding_value=0.0)
    tgt = pad_sequence(tokens, batch_first=True, padding_value=PAD_ID)

    src_key_padding_mask = (
        torch.arange(src.size(1))[None, :] >= src_lengths[:, None]
    )

    tgt_in = tgt[:, :-1]
    tgt_out = tgt[:, 1:]
    tgt_key_padding_mask = tgt_in.eq(PAD_ID)

    tgt_length = tgt_in.size(1)
    causal_mask = torch.triu(
        torch.ones((tgt_length, tgt_length), dtype=torch.bool),
        diagonal=1
    )

    return (
        src,
        src_key_padding_mask,
        tgt_in,
        tgt_out,
        tgt_key_padding_mask,
        causal_mask
    )


train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn
)

# 使用固定順序的 DataLoader 進行示範，方便每次確認同一筆樣本
demo_loader = DataLoader(
    train_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn
)

batch = next(iter(demo_loader))
names = [
    'src', 'src_key_padding_mask', 'tgt_in', 'tgt_out',
    'tgt_key_padding_mask', 'causal_mask'
]
for name, tensor in zip(names, batch):
    print(name, tuple(tensor.shape), tensor.dtype)

print('第一筆影片:', train_dataset.rows[0]['video_id'])
print('第一筆中文:', train_dataset.rows[0]['text'][:120])

src (1, 736, 128) torch.float32
src_key_padding_mask (1, 736) torch.bool
tgt_in (1, 67) torch.int64
tgt_out (1, 67) torch.int64
tgt_key_padding_mask (1, 67) torch.bool
causal_mask (67, 67) torch.bool
第一筆影片: part_1_1
第一筆中文: 看新聞 開視野 歡迎收看今天6月6號的公視手語新聞 首先來關心台南市永康鬧區 昨晚突然飄出瓦斯異味 臺南市警消獲報到場偵測到可燃性氣體 緊急疏散周邊店家人員 封路布水線防護 瓦斯公司也關閉閥門 尋找洩漏源 由於今天是上班日 要看搶修進度 才


## 3. 建立 Transformer Encoder-Decoder

架構對應原本《Sign Language Transformer: Implementation Plan》的小型模型建議：

1. `Linear(128 → d_model)` + Positional Encoding
2. Transformer Encoder：學習各 Frame 之間的動作關係
3. Transformer Decoder：結合已知文字與 Encoder 特徵（Cross-Attention）
4. `Linear(d_model → vocab_size)`：輸出每個文字位置的詞彙預測分數

`MAX_LEN=4096` 用來容納預期的 15 FPS 長序列；這**不代表**所有這麼長的序列都一定能放進 GPU 記憶體。

In [4]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=4096):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1)
        div = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class SignTransformer(nn.Module):
    def __init__(
        self, vocab_size, kp_dim=128, d_model=192, nhead=4,
        num_encoder_layers=3, num_decoder_layers=3,
        dim_ff=512, dropout=0.2, pad_id=0, max_len=4096
    ):
        super().__init__()

        self.input_proj = nn.Linear(kp_dim, d_model)
        self.tgt_embed = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos_enc = PositionalEncoding(d_model, max_len=max_len)

        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=nhead,
                dim_feedforward=dim_ff,
                dropout=dropout,
                batch_first=True
            ),
            num_layers=num_encoder_layers
        )

        self.decoder = nn.TransformerDecoder(
            nn.TransformerDecoderLayer(
                d_model=d_model,
                nhead=nhead,
                dim_feedforward=dim_ff,
                dropout=dropout,
                batch_first=True
            ),
            num_layers=num_decoder_layers
        )

        self.output_proj = nn.Linear(d_model, vocab_size)

    def forward(
        self, src, tgt_in, src_key_padding_mask,
        tgt_key_padding_mask, causal_mask
    ):
        src = self.pos_enc(self.input_proj(src))
        memory = self.encoder(
            src, src_key_padding_mask=src_key_padding_mask
        )

        tgt = self.pos_enc(self.tgt_embed(tgt_in))
        decoded = self.decoder(
            tgt,
            memory,
            tgt_mask=causal_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=src_key_padding_mask
        )
        return self.output_proj(decoded)


model = SignTransformer(
    vocab_size=vocab_size,
    d_model=D_MODEL,
    nhead=NHEAD,
    num_encoder_layers=ENCODER_LAYERS,
    num_decoder_layers=DECODER_LAYERS,
    dim_ff=DIM_FF,
    dropout=DROPOUT,
    pad_id=PAD_ID,
    max_len=MAX_LEN
).to(device)

print('模型參數數量:', f'{sum(p.numel() for p in model.parameters()):,}')

模型參數數量: 3,286,076


## 4. 第一次 Forward Pass（不更新模型參數）

這一步只檢查 Encoder / Decoder 是否能成功運算。

正確的輸出格式應為 `(Batch Size, Target Length, Vocabulary Size)`；Vocabulary Size 取自你自己的 `vocab.json`，不是固定 2000。

In [5]:
src, src_kpm, tgt_in, tgt_out, tgt_kpm, causal = [
    tensor.to(device) for tensor in batch
]

model.eval()
with torch.no_grad():
    logits = model(src, tgt_in, src_kpm, tgt_kpm, causal)

print('Device:', device)
print('Input Shape:', tuple(src.shape))
print('Target Shape:', tuple(tgt_in.shape))
print('Output Shape:', tuple(logits.shape))
print('Vocabulary Size:', vocab_size)
print('輸出維度正確:', logits.shape == (src.size(0), tgt_in.size(1), vocab_size))

c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\nn\modules\transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\NestedTensorImpl.cpp:182.)
  output = torch._nested_tensor_from_mask(


Device: cuda
Input Shape: (1, 736, 128)
Target Shape: (1, 67)
Output Shape: (1, 67, 1916)
Vocabulary Size: 1916
輸出維度正確: True


## 5. 確認初始 Loss

使用 Cross-Entropy Loss，比較 Decoder 輸出與 `tgt_out`。由於還沒有訓練，Loss 不會有特別低的期待值。

In [6]:
criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_ID,
    label_smoothing=0.1
)

with torch.no_grad():
    initial_loss = criterion(
        logits.reshape(-1, logits.size(-1)),
        tgt_out.reshape(-1)
    )

print('初始 Loss:', f'{initial_loss.item():.4f}')

初始 Loss: 7.8755


## 6. 可選：測試一次反向傳播（預設關閉）

如果要確認梯度和 AdamW 能正常運作，把 `RUN_ONE_TRAIN_STEP = True`。它只更新 **1 個 Batch**，不是正式訓練。長影片可能需要較多 GPU 記憶體，請先確保 Forward Pass 成功。

In [ ]:
RUN_ONE_TRAIN_STEP = True

if RUN_ONE_TRAIN_STEP:
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
    model.train()

    optimizer.zero_grad()
    logits_train = model(src, tgt_in, src_kpm, tgt_kpm, causal)
    train_loss = criterion(
        logits_train.reshape(-1, logits_train.size(-1)),
        tgt_out.reshape(-1)
    )
    train_loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    print('單步訓練 Loss:', f'{train_loss.item():.4f}')
else:
    print('已略過單步反向傳播。')

已略過單步反向傳播。


## 下一步

如果第 4～5 節成功，就代表資料流與 Transformer 的 Forward Pass 已經接上。下一步建議先進行 **5～10 筆資料的 Overfitting Test**，確認模型可以逐步記住少量訓練資料，接著才考慮完整訓練、Validation Loss、Checkpoint 與 Early Stopping。

**提醒**：你目前使用的是「整支影片 → 整段中文字」配對，最長約 150 秒，部分樣本的長度可能比這次抽中的 Sample 更長。成功執行一筆 Forward Pass 不代表所有長度都能在現有記憶體下成功訓練，也不代表翻譯效果已經驗證。